# Lesson 06a Walkthrough: Why Good Database Design Matters

**Course:** Database Applications Development  
**Medina County Career Center**

---

In this walkthrough, we're going to look under the hood of the IMDb database. We'll answer: **Why is this database split into separate tables instead of one giant table?**

By the end, you'll understand three key concepts:
1. **Primary Keys** — how each row gets a unique ID
2. **Foreign Keys** — how tables link together  
3. **Redundancy** — why storing duplicate data is a problem

**Note:** Refer to the Data Dictionary (`imdb_data_dictionary.md`) for friendly names and detailed column descriptions.

# Lesson 06a Walkthrough: Why Good Database Design Matters

**Course:** Database Applications Development  
**Medina County Career Center**

---

In this walkthrough, we're going to look under the hood of the IMDb database. We'll answer: **Why is this database split into separate tables instead of one giant table?**

By the end, you'll understand three key concepts:
1. **Primary Keys** — how each row gets a unique ID
2. **Foreign Keys** — how tables link together  
3. **Redundancy** — why storing duplicate data is a problem

**Note:** Refer to the Data Dictionary (`imdb_data_dictionary.md`) for friendly names and detailed column descriptions.

In [1]:
# Setup — import libraries and connect to database
import pandas as pd
import sqlite3

dbPath = 'imdb_class_2010plus.db'
connection = sqlite3.connect(dbPath)
cursor = connection.cursor()

print(f'Connected to: {dbPath}')

Connected to: imdb_class_2010plus.db


---

## Part 1: What Are Primary Keys?

A **primary key** is a column (or set of columns) that uniquely identifies each row in a table. No two rows can have the same primary key value.

Think of it like your student ID — no two students share the same number. That's exactly what a primary key does for a database row.

Let's see the primary keys in the **Titles — Movies & TV Shows** table (`title_basics`).

In [2]:
# -------
# Prove that Title ID (tconst) is unique in title_basics
# -------
# If total rows = unique tconst values, then every row has a different ID

query = '''
    SELECT 
        COUNT(*) as totalRows,
        COUNT(DISTINCT tconst) as uniqueTconst
    FROM title_basics
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))
print()

total = df['totalRows'][0]
unique = df['uniqueTconst'][0]
if total == unique:
    print(f'Success! All {total:,} rows have a unique Title ID (tconst).')
    print('tconst works as a primary key.')
else:
    print(f'Problem: {total:,} rows but only {unique:,} unique IDs.')

 totalRows  uniqueTconst
     19462         19462

Success! All 19,462 rows have a unique Title ID (tconst).
tconst works as a primary key.


In [6]:
# -------
# Why can't we use the title name as the primary key?
# -------
# Find titles (primaryTitle) that appear more than once

query = '''
    SELECT primaryTitle, COUNT(*) as count
    FROM title_basics
    GROUP BY primaryTitle
    HAVING count > 1
    ORDER BY count DESC
    LIMIT 12
'''

df = pd.read_sql_query(query, connection)
print('Titles that appear more than once:')
print(df.to_string(index=False))
print()
print('Multiple movies & TV shows share the same name!')
print('That\'s why we use Title ID (tconst) instead — it\'s always unique.')

Titles that appear more than once:
primaryTitle  count
   Aftermath      6
     Monster      5
       Blind      5
        Wolf      4
      Utopia      4
       Trust      4
        Prey      4
    Paradise      4
        Love      4
        Home      4
        Gold      4
        Eden      4

Multiple movies & TV shows share the same name!
That's why we use Title ID (tconst) instead — it's always unique.


The same principle applies to the **People — Actors, Directors & More** table (`name_basics`): The **Person ID** (`nconst`) is the primary key. Multiple people can share the same name (like "David Smith" or "Michael Johnson"), so person names can't be primary keys. Person IDs can be.

### Primary Key Summary

| Friendly Name | Table Name | Primary Key | Example |
|---|---|---|---|
| Titles — Movies & TV Shows | `title_basics` | **Title ID** (`tconst`) | tt1375666 (Inception) |
| Movie Ratings | `title_ratings` | **Title ID** (`tconst`) | tt1375666 |
| People — Actors, Directors & More | `name_basics` | **Person ID** (`nconst`) | nm0000138 (Leonardo DiCaprio) |
| Cast & Crew Credits | `title_principals` | **Title ID** + **Ordering** (`tconst` + `ordering`) | tt1375666, 1 |
| Crew Credits | `title_crew_person` | **Title ID** + **Person ID** + **Role** (`tconst` + `nconst` + `role`) | tt1375666, nm0634240, director |

Notice that the last two tables use **composite keys** — two or more columns together form the unique identifier. That's because one movie has many actors, and one actor is in many movies.

---

## Part 2: What Are Foreign Keys?

A **foreign key** is a column in one table that **references** the primary key of another table. It's the link that connects tables together.

In our database:
- `title_principals.tconst` is a **foreign key** pointing to `title_basics.tconst`  
- `title_principals.nconst` is a **foreign key** pointing to `name_basics.nconst`

The foreign key says: "Go look up this ID in that other table to get the full details."

In [7]:
# -------
# See how foreign keys work: raw data from Cast & Crew Credits (title_principals)
# -------
# This shows just the IDs — not very readable!

query = '''
    SELECT tconst, nconst, category
    FROM title_principals
    WHERE tconst = 'tt1375666'
    ORDER BY ordering
    LIMIT 5
'''

df = pd.read_sql_query(query, connection)
print('Raw Cast & Crew Credits data (just IDs):')
print(df.to_string(index=False))

Raw Cast & Crew Credits data (just IDs):
   tconst    nconst category
tt1375666 nm0000138    actor
tt1375666 nm0330687    actor
tt1375666 nm0680983    actor
tt1375666 nm0913822    actor
tt1375666 nm0362766    actor


In [1]:
# -------
# Now JOIN using foreign keys to get actual names
# -------
# The foreign keys let us "follow the links" to other tables

query = '''
    SELECT b.primaryTitle,
           n.primaryName,
           p.category
    FROM title_principals p
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE p.tconst = 'tt1375667'
    ORDER BY p.ordering
    LIMIT 5
'''

df = pd.read_sql_query(query, connection)
print('Same data, with real names (using JOINs on foreign keys):')
print(df.to_string(index=False))
print()
print('The foreign keys connected the IDs to their readable values.')
print('Foreign keys are the links that make it all work!')

NameError: name 'pd' is not defined

In [ ]:
# -------
# Verify that every foreign key points to a valid primary key
# -------
# Does every Title ID in title_principals exist in title_basics?

query = '''
    SELECT COUNT(*) as orphanedRows
    FROM title_principals p
    WHERE p.tconst NOT IN (SELECT tconst FROM title_basics)
'''

df = pd.read_sql_query(query, connection)
orphans = df['orphanedRows'][0]
print(f'Credits pointing to non-existent titles: {orphans}')

if orphans == 0:
    print('Every foreign key points to a valid title. The links are solid!')
else:
    print(f'Warning: {orphans} broken links found!')

### How Tables Connect: Foreign Key Map

Let's take a look at the `imdb_schema_chart.md`

---

## Part 3: Why Redundancy Is the Enemy

Now for the big payoff — let's see what would happen if we **didn't** split the data into separate tables.

Imagine one giant table where every credit row also stores the full movie title, year, rating, and actor's birth year and profession. That's redundancy — storing the same fact over and over.

In [ ]:
# -------
# How much redundancy would we have in a bad design?
# -------

query = '''
    SELECT 
        COUNT(*) as totalCredits,
        COUNT(DISTINCT tconst) as uniqueTitles
    FROM title_principals
'''

df = pd.read_sql_query(query, connection)
totalCredits = df['totalCredits'][0]
uniqueTitles = df['uniqueTitles'][0]

print(f'Total credits in Cast & Crew Credits: {totalCredits:,}')
print(f'Unique titles: {uniqueTitles:,}')
print()
print('In a BAD design (one giant table):')
print(f'  Title info would be stored {totalCredits:,} times')
print()
print('In our GOOD design (separate tables):')
print(f'  Title info is stored {uniqueTitles:,} times (once each)')
print()
redundancy = totalCredits - uniqueTitles
print(f'Storage savings: {redundancy:,} duplicate rows eliminated!')

In [ ]:
# -------
# The Update Problem: redundancy makes changes dangerous
# -------
# If title data was in every row, how many rows would we need to update?

query = '''
    SELECT b.primaryTitle, COUNT(*) as rowsToUpdate
    FROM title_principals p
    JOIN title_basics b ON p.tconst = b.tconst
    WHERE b.primaryTitle IN (
        'Inception', 'Interstellar', 'The Batman', 
        'Avengers: Endgame', 'Spider-Man: No Way Home'
    )
    GROUP BY b.primaryTitle
    ORDER BY rowsToUpdate DESC
'''

df = pd.read_sql_query(query, connection)
print('If title info was in every credit row (BAD design):')
print('Changing one movie title would require updating:')
print()
print(df.to_string(index=False))
print()
print('In our GOOD design: just 1 row each in Titles — Movies & TV Shows!')
print('Change the title once, and all credits automatically show the new name.')

In [ ]:
# -------
# The Typo Problem: redundancy creates inconsistent data
# -------

print('In a BAD design, the same movie might be spelled differently:')
print()
print('  Row 1: Spider-Man: No Way Home    | Tom Holland  | actor')
print('  Row 2: Spider-Man: No Way Home    | Zendaya      | actress')
print('  Row 3: Spiderman: No Way Home     | Benedict C.  | actor    ← TYPO!')
print('  Row 4: Spider-Man: No Way Home    | Jacob B.     | actor')
print()
print('Now searches for "Spider-Man: No Way Home" miss Row 3!')
print()
print('In our GOOD design: the title is stored ONCE in Titles table.')
print('All credits reference the same ID. No chance of typos!')
print('Database integrity is preserved.')

---

## Recap: The Three Rules of Good Database Design

**1. Primary Keys — Every table has a unique identifier**
- **Titles — Movies & TV Shows** (`title_basics`) → **Title ID** (`tconst`)
- **People — Actors, Directors & More** (`name_basics`) → **Person ID** (`nconst`)
- Names and titles are NOT unique enough to be primary keys. IDs are better.

**2. Foreign Keys — Tables connect through ID references**
- `title_principals.tconst` references `title_basics.tconst`
- `title_principals.nconst` references `name_basics.nconst`
- JOINs use foreign keys to pull data from multiple tables without duplication

**3. No Redundancy — Each fact is stored exactly once**
- "Inception" appears once in **Titles** table, not in every credit row
- "Leonardo DiCaprio" appears once in **People** table, not in every movie
- We save hundreds of thousands of duplicate copies
- Updates are safe, and typos can't create data inconsistency

**Next Steps:**
- Open the task notebook to practice these concepts
- Refer to the study guide for deeper dives into each topic

In [ ]:
# Clean up
connection.close()
print('Database connection closed.')